In [1]:
! pip install -U datasets fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
! pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 34.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [8]:
# Dataset

from datasets import load_dataset
import torch
import torch.nn.functional as F

dataset=load_dataset("imdb")
small_train = dataset['train'].shuffle(seed=42).select([i for i in list(range(1000))])
small_test = dataset['test'].shuffle(seed=42).select([i for i in list(range(300))])

In [16]:
# So'zlarni raqamga o'tkazish

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True,padding='max_length',max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [17]:
# Model

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased',num_labels=2)

from transformers import TrainingArguments,Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir = './logs',
    logging_steps=10,
    report_to='none')

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

trainer.evaluate()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
10,0.698978
20,0.670348
30,0.659657
40,0.522716
50,0.414306
60,0.411936
70,0.383853
80,0.229088
90,0.250272
100,0.275219


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.43666544556617737,
 'eval_runtime': 2.1095,
 'eval_samples_per_second': 142.211,
 'eval_steps_per_second': 9.007,
 'epoch': 3.0}

In [18]:

texts = [
    'This movie good',
    'This movie was absolutely lovely',
    'This movie was absolutely awful',
    'Just ok movie not so bad'
]

inputs = tokenizer(texts, padding=True, truncation=True,max_length=256, return_tensors="pt").to('cuda')

with torch.no_grad():
  outputs = model(**inputs)
  probs = F.softmax(outputs.logits, dim=1)
  predictions = torch.argmax(probs, dim=1)

label_map = {0: 'negative', 1: 'positive'}

for text, pred,prob in zip(texts, predictions,probs):
  print(text,label_map[pred.item()],prob[pred.item()])

This movie good positive tensor(0.9408, device='cuda:0')
This movie was absolutely lovely positive tensor(0.9854, device='cuda:0')
This movie was absolutely awful negative tensor(0.9864, device='cuda:0')
Just ok movie not so bad negative tensor(0.9788, device='cuda:0')
